# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import duckdb, os
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
content = f"read_parquet('{REL}/dim_content.parquet')"

In [3]:
staleness_check = con.sql(f"""
    SELECT
        CASE
            WHEN DATEDIFF('day', c.content_created_date, DATE '2026-03-31') < 90 THEN 'fresh (<90d)'
            WHEN DATEDIFF('day', c.content_created_date, DATE '2026-03-31') < 180 THEN 'moderate (90-180d)'
            ELSE 'stale (180d+)'
        END as staleness_bucket,
        AVG(f.gsc_impressions) as avg_impressions,
        COUNT(*) as n
    FROM {fact} f JOIN {content} c ON f.content_hash_id = c.content_hash_id
    GROUP BY 1
    ORDER BY 1
""").df()
print(staleness_check)

     staleness_bucket  avg_impressions        n
0        fresh (<90d)        37.851194  2116433
1  moderate (90-180d)        50.104480  1133357
2       stale (180d+)        21.809890  6591588


**Verdict: MIXED.**

Staleness alone does not cleanly predict lower traffic —
the relationship isn't monotonic (moderate-age pages actually average MORE
impressions than fresh ones, 50.10 vs 37.85). The stale bucket does have the
lowest per-page average (21.81), consistent with decay, but because it's by
far the largest bucket (6.59M rows), it still contains the most total
impression volume overall. This is actually useful evidence for the rule
design: most stale pages genuinely get low traffic, but the stale-AND-visible
subset the rule targets (impressions_march >= 500) is a real, meaningful
minority worth isolating — staleness alone isn't a strong enough signal on
its own, which is exactly why the rule requires it combined with a visibility
threshold, not staleness in isolation. n = 9,841,378 total — plenty large.

In [4]:
ctr_position_check = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 10 THEN 'top 10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            ELSE '20+'
        END as position_bucket,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr,
        COUNT(*) as n
    FROM {fact}
    WHERE gsc_impressions >= 10
    GROUP BY 1
    ORDER BY 1
""").df()
print(ctr_position_check)

  position_bucket   avg_ctr        n
0           11-20  0.002614   358200
1             20+  0.001305   445678
2          top 10  0.003348  1343651


**Verdict: CONFIRMED.** 

CTR drops monotonically as position worsens (top 10
> 11-20 > 20+), matching the CTR-fix logic from the session. n = 2,147,529
total. One honest caveat: the absolute CTR values are lower than typical GSC
benchmarks (0.33% at top 10, where 20-30%+ is more typical) — this doesn't
break the directional finding, but suggests either a low-quality inventory
mix or that the impressions >= 10 filter is still letting in a lot of
low-intent/low-CTR query matches. Worth a tighter volume filter in a future
iteration.

**My rule, in plain words:** flag a page for review if it is stale
(180+ days since update) AND still visible (500+ impressions), scored by
how much traffic it's still pulling.

**Reason codes this rule can output:**
- `stale_visible_page` — stale and still getting real traffic

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
scored = con.sql(f"""
    SELECT f.content_hash_id,
           SUM(f.gsc_impressions) as impressions_march,
           AVG(f.gsc_avg_position) as avg_position,
           DATEDIFF('day', c.content_created_date, DATE '2026-03-31') as content_age_days
    FROM {fact} f JOIN {content} c ON f.content_hash_id = c.content_hash_id
    GROUP BY 1, c.content_created_date
""").df()

scored["score"] = (
    (scored["content_age_days"] >= 180).astype(int) * 0.5 +
    (scored["impressions_march"] >= 500).astype(int) * 0.5
) * scored["impressions_march"]

scored["reason_code"] = "stale_visible_page"
scored["action"] = "review_for_refresh"

queue = scored.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
queue.head(10)

,content_hash_id,impressions_march,avg_position,content_age_days,score,reason_code,action
106936,content_eadb33b5df496f4a,617124.0,2.383011,375,617124.0,stale_visible_page,review_for_refresh
150096,content_ec2e0346994fb5a5,245276.0,2.854514,434,245276.0,stale_visible_page,review_for_refresh
111076,content_e8a52cf3d5988c07,244931.0,15.008339,230,244931.0,stale_visible_page,review_for_refresh
241775,content_0e03de7680314cd5,221310.0,2.675217,375,221310.0,stale_visible_page,review_for_refresh
62544,content_e7b5dd4dff461ad2,205045.0,4.544203,343,205045.0,stale_visible_page,review_for_refresh
138169,content_8d7d99f109e19aa2,203497.0,2.563756,375,203497.0,stale_visible_page,review_for_refresh
194007,content_36e53e9c707674fc,194579.0,32.766674,229,194579.0,stale_visible_page,review_for_refresh
220974,content_4ffe18112a5642e3,186983.0,2.331060,375,186983.0,stale_visible_page,review_for_refresh
217719,content_471d9cabce329a66,164885.0,4.656030,375,164885.0,stale_visible_page,review_for_refresh
175872,content_512dbad65bd5ade9,154358.0,3.019798,187,154358.0,stale_visible_page,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

**Note:** below are the top 10 rows currently pulled (`queue.head(10)`). To
get the full top-20 as the skeleton asks, change this to `queue.head(20)`
and re-run — the same review pattern applies to rows 11-20.

1. **content_eadb33b5df496f4a** — action: review_for_refresh; reason:
   stale_visible_page. Stale (375 days) and the highest traffic in the whole
   queue (617,124 impressions), with a strong position (2.38). Confidence:
   high — clears both thresholds by a wide margin. Would be wrong if this
   page was manually updated recently but our staleness field wasn't
   refreshed to reflect it (a data-lag risk, not a rule-logic risk).

2. **content_ec2e0346994fb5a5** — action: review_for_refresh; reason:
   stale_visible_page. Stale (434 days), 245,276 impressions, strong position
   (2.85). Confidence: high — comfortably clears both thresholds. Would be
   wrong under the same staleness-lag risk as row 1.

3. **content_e8a52cf3d5988c07** — action: review_for_refresh; reason:
   stale_visible_page. Stale (230 days), 244,931 impressions, but position is
   15.0 — much weaker than most of this list. Confidence: medium. This page
   may need a ranking/intent fix rather than a content refresh — the action
   label itself may be imprecise here, not just the score.

4. **content_0e03de7680314cd5** — action: review_for_refresh; reason:
   stale_visible_page. Stale (375 days), 221,310 impressions, strong position
   (2.68). Confidence: high. Would be wrong under the same staleness-lag risk.

5. **content_e7b5dd4dff461ad2** — action: review_for_refresh; reason:
   stale_visible_page. Stale (343 days), 205,045 impressions, good position
   (4.54). Confidence: high. Same staleness-lag caveat applies.

6. **content_8d7d99f109e19aa2** — action: review_for_refresh; reason:
   stale_visible_page. Stale (375 days), 203,497 impressions, strong position
   (2.56). Confidence: high. Same staleness-lag caveat applies.

7. **content_36e53e9c707674fc** — action: review_for_refresh; reason:
   stale_visible_page. Stale (229 days), 194,579 impressions, but position is
   32.77 — the weakest position in the entire top 10. Confidence: low. A page
   averaging position 33 rarely earns 194K impressions unless it's an average
   across many queries where one performs very well and drags the rest down —
   worth inspecting the underlying query mix before trusting this one as-is.

8. **content_4ffe18112a5642e3** — action: review_for_refresh; reason:
   stale_visible_page. Stale (375 days), 186,983 impressions, strong position
   (2.33) — the best position in the top 10. Confidence: high. Same
   staleness-lag caveat applies.

9. **content_471d9cabce329a66** — action: review_for_refresh; reason:
   stale_visible_page. Stale (375 days), 164,885 impressions, good position
   (4.66). Confidence: high. Same staleness-lag caveat applies.

10. **content_512dbad65bd5ade9** — action: review_for_refresh; reason:
    stale_visible_page. Stale (187 days) — just barely past the 180-day
    threshold — 154,358 impressions, strong position (3.02). Confidence:
    medium. This is the borderline case of the group: it clears the staleness
    threshold by only 7 days, so a small correction to the staleness field or
    a slightly different cutoff could easily drop it out of the top 10.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

**Weak picks — which of my top-10 look shakiest, and why:**

- **Row 3 (content_e8a52cf3d5988c07)** and **Row 7 (content_36e53e9c707674fc)**
  both have high scores but weak positions (15.0 and 32.77). Since my score
  formula multiplies the binary thresholds by raw impressions only, it never
  actually weighs position — so a page ranking poorly can still score very
  high just because it has volume. Row 7 is the weakest of all ten: position
  32.77 with 194,579 impressions is an unusual combination, and it's worth
  checking the underlying query mix before trusting it as a real "quick
  refresh" candidate rather than a ranking problem in disguise.

- **Row 10 (content_512dbad65bd5ade9)** is a borderline pick, not a wrong
  one — it clears the staleness threshold by only 7 days (187 vs the 180-day
  cutoff). A small correction to the staleness field, or a slightly stricter
  threshold, would drop it out of the top 10 entirely. It's the pick I'd
  trust least to be stable if I reran this next month.

- **General weakness of the rule itself:** every top-10 score equals its
  impressions_march exactly, because both threshold checks are binary
  (0 or 1) and get multiplied together. This means the rule can filter
  candidates but can't actually rank them by anything other than raw
  traffic — two pages with identical impressions but very different
  positions or CTR would score identically. This is the kind of gap a
  learned model should be able to close.

**Leakage check:**

- No FlyRank product flags used anywhere — `health_score`, `priority_score`,
  `action_type`, and `refresh_tier` were never pulled from any table or used
  in scoring, consistent with the observable-only rule for this internship.
- No future-window data used — every feature (`impressions_march`,
  `avg_position`, `content_age_days`) is computed strictly from March 2026
  data or from `content_created_date`, a fixed past field. Nothing from
  April or later touches this notebook.
- No label-derived inputs — this is a baseline rule, not a trained model, so
  there's no label to leak in the first place. The one thing to watch when
  this baseline gets compared against a real model next week: the model's
  features must follow the same discipline (prior-window only).

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.